In [0]:
%run ../Databricks-Bookstore-Engineering/Copy-Datasets

In [0]:
# %run ../Databricks-Bookstore-Engineering/Copy-Datasets

current_catalog = spark.sql("SELECT current_catalog()").collect()[0][0]
dataset_bookstore = f"/Volumes/{current_catalog}/bookstore_eng_pro/dataset"
checkpoint_path = f"/Volumes/{current_catalog}/bookstore_eng_pro/checkpoints"
print(dataset_bookstore)

In [0]:
%sql
Alter table orders_silver add constraint timestamp_within_range CHECK (order_timestamp >= '2020-01-01');

In [0]:
%sql

DESCRIBE EXTENDED orders_silver

In [0]:
%sql
INSERT INTO orders_silver
VALUES ('1', '2022-02-01 00:00:00.000', 'C00001', 0, 0, NULL),
       ('2', '2019-05-01 00:00:00.000', 'C00001', 0, 0, NULL),
       ('3', '2023-01-01 00:00:00.000', 'C00001', 0, 0, NULL)

In [0]:
%sql
ALTER TABLE orders_silver ADD CONSTRAINT valid_quantity CHECK (quantity > 0);

In [0]:
%sql

DESCRIBE EXTENDED orders_silver

In [0]:
%sql
drop table orders_silver

In [0]:
from pyspark.sql import functions as F

json_schema = "order_id STRING, order_timestamp Timestamp, customer_id STRING, quantity BIGINT, total BIGINT, books ARRAY<STRUCT<book_id STRING, quantity BIGINT, subtotal BIGINT>>"

query = (spark.readStream.table("bronze")
        .filter("topic = 'orders'")
        .select(F.from_json(F.col("value").cast("string"), json_schema).alias("v"))
        .select("v.*")
        .filter("quantity > 0")
     .writeStream
        .option("checkpointLocation", f"{checkpoint_path}/orders_silver1=2")
        .trigger(availableNow=True)
        .table("orders_silver"))

query.awaitTermination()

In [0]:
%sql
SELECT *
FROM orders_silver
where quantity <= 0

In [0]:
%sql
describe extended orders_silver

In [0]:
%sql
drop table orders_silver